In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from ipywidgets import interact_manual, widgets

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# ==========================================
# DATA LOAD
# ==========================================

csv_filename = 'advertising.csv'

if not os.path.exists(csv_filename):
    raise FileNotFoundError(f"Error: '{csv_filename}' not found.")

df = pd.read_csv(csv_filename)
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

In [ ]:
# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# ==========================================
# MODEL TRAINING (Ridge Regression)
# ==========================================

print("Training Ridge Regression Model (L2 Regularization)...")
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

y_pred = ridge_model.predict(X_test)
print(f"\n--- METRICS ---\nR2 Score: {r2_score(y_test, y_pred):.4f}\nMAE: {mean_absolute_error(y_test, y_pred):,.2f} Units\n")

In [ ]:
# ==========================================
# EVALUATION & VISUALIZATIONS
# ==========================================

sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Chart 1: Financial Correlation Heatmap
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            vmax=1, center=0, square=True, linewidths=.5, ax=axes[0])
axes[0].set_title('Budget vs. Sales Correlation Map', fontsize=14, fontweight='bold')

# Chart 2: Residual Error Plot
residuals = y_test - y_pred
sns.histplot(residuals, kde=True, color="purple", ax=axes[1])
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_title('Model Error Distribution (Residuals)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Prediction Error')

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# INTERACTIVE UI: Test the Model
# ==========================================

print("\n📊 Test the Ridge Marketing Model 📊")
@interact_manual(
    TV=widgets.FloatSlider(min=0, max=300, value=150, description="TV ($):"),
    Radio=widgets.FloatSlider(min=0, max=50, value=25, description="Radio ($):"),
    News=widgets.FloatSlider(min=0, max=150, value=20, description="News ($):")
)
def predict_sales_ridge(TV, Radio, News):
    try:
        feats = pd.DataFrame([[TV, Radio, News]], columns=['TV', 'Radio', 'Newspaper'])
        print(f"\n>> 📦 Projected Sales Volume: {ridge_model.predict(feats)[0]:,.2f} Units")
    except Exception as e:
        print(f"Error: {e}")